In [28]:
from qiskit import QuantumCircuit
%pip install nbformat

%run testing_framework.ipynb

Note: you may need to restart the kernel to use updated packages.


In [29]:
import os

print("Current directory:", os.getcwd())
print("Files:", os.listdir())


Current directory: c:\Users\robbi\OneDrive\Desktop\Fall 27\capstone\testing_framework\QUBIT_Capstone
Files: ['.git', '.gitignore', 'quantum_gate_algorithm_library.ipynb', 'README.md', 'requirements.txt', 'testing_framework.ipynb', 'testing_harness.ipynb']


In [30]:
def identity_circuit():
    qc = QuantumCircuit(1)
    return qc


def x_gate_circuit():
    qc = QuantumCircuit(1)
    qc.x(0)
    return qc


def hadamard_circuit():
    qc = QuantumCircuit(1)
    qc.h(0)
    return qc

In [31]:
test_cases = [
    {
        "name": "Valid X Gate",
        "qbits": x_gate_circuit(),
        "expected_output": "1",
        "backend": "aer_simulator",
        "shots": 1000,
        "accuracy": 0.90,
        "expected_result": True
    },
    {
        "name": "Wrong Expected Output",
        "qbits": x_gate_circuit(),
        "expected_output": "0",
        "backend": "aer_simulator",
        "shots": 1000,
        "accuracy": 0.90,
        "expected_result": False
    },
    {
        "name": "Hadamard Below Threshold",
        "qbits": hadamard_circuit(),
        "expected_output": "0",
        "backend": "aer_simulator",
        "shots": 1000,
        "accuracy": 0.90,
        "expected_result": False
    },
    {
        "name": "Invalid Backend Type",
        "qbits": x_gate_circuit(),
        "expected_output": "1",
        "backend": 123,
        "expected_error": TypeError
    },
    {
        "name": "Empty Backend Name",
        "qbits": x_gate_circuit(),
        "expected_output": "1",
        "backend": "",
        "expected_error": ValueError
    },
    {
        "name": "Negative Shot Count",
        "qbits": x_gate_circuit(),
        "expected_output": "1",
        "backend": "aer_simulator",
        "shots": -100,
        "expected_error": ValueError
    },
    {
        "name": "Invalid Expected Output",
        "qbits": x_gate_circuit(),
        "expected_output": "2",
        "backend": "aer_simulator",
        "expected_error": ValueError
    },
    {
        "name": "Invalid Accuracy",
        "qbits": x_gate_circuit(),
        "expected_output": "1",
        "backend": "aer_simulator",
        "accuracy": 1.5,
        "expected_error": ValueError
    }
]

In [32]:
import os
from dotenv import load_dotenv
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService


def get_backend(backend_name):
    """
    @brief Validates and retrieves a quantum computing backend.

    @param backend_name Name of the requested backend.
    @return Dictionary containing backend object, type, and name.
    """

    # Validate backend name
    if not isinstance(backend_name, str):
        raise TypeError("Backend name must be a string.")

    backend_name = backend_name.strip()

    if not backend_name:
        raise ValueError("Backend name cannot be empty.")

    # Local simulator - no credentials or confirmation needed
    if backend_name == "aer_simulator":
        return {
            "backend": AerSimulator(),
            "type": "simulator",
            "name": backend_name
        }

    # Confirm hardware selection before accessing credentials
    confirmation = input(
        f"You selected to use the quantum computer '{backend_name}'. "
        "Are you sure you want to continue? (Y/N): "
    ).strip().upper()

    if confirmation != "Y":
        raise PermissionError(
            "Quantum computer selection cancelled."
        )

    # Load IBM Quantum credentials
    load_dotenv()

    token = os.getenv("IBM_QUANTUM_API_KEY")
    instance = os.getenv("IBM_QUANTUM_INSTANCE")

    if not token or not instance:
        raise ValueError(
            "IBM Quantum credentials are missing."
        )

    # Connect to IBM Quantum and validate backend
    try:
        service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=token,
            instance=instance
        )

        backend = service.backend(backend_name)

    except Exception as error:
        raise ValueError(
            f"Unable to access backend '{backend_name}'."
        ) from error

    # Verify selected backend is real hardware
    if backend.configuration().simulator:
        raise ValueError(
            "Selected backend is a simulator, not a real quantum computer."
        )

    # Verify backend is operational
    if not backend.status().operational:
        raise RuntimeError(
            f"Backend '{backend_name}' is not operational."
        )

    # Return validated hardware backend
    return {
        "backend": backend,
        "type": "hardware",
        "name": backend_name
    }

In [33]:
results = testing_harness(test_cases)

passed = sum(test["passed"] for test in results)
total = len(results)

print("\nTesting Harness Summary")
print(f"Passed: {passed}/{total}")
print(f"Failed: {total - passed}/{total}")

FAIL: Valid X Gate — NameError: name 'transpile' is not defined
FAIL: Wrong Expected Output — NameError: name 'transpile' is not defined
FAIL: Hadamard Below Threshold — NameError: name 'transpile' is not defined
PASS: Invalid Backend Type — TypeError: Backend name must be a string.
PASS: Empty Backend Name — ValueError: Backend name cannot be empty.
PASS: Negative Shot Count — ValueError: Shots must be a positive integer.
PASS: Invalid Expected Output — ValueError: Expected output must contain only 0 and 1.
PASS: Invalid Accuracy — ValueError: Accuracy must be between 0 and 1.

Testing Harness Summary
Passed: 5/8
Failed: 3/8
